In [20]:
import pandas as pd
import polars as pl
import numpy as np
from pathlib import Path

from polars import selectors as cs

schedule = Path('data/schedule')
statcast = Path('data/statcast')

In [21]:
!nvidia-smi

Wed May 27 12:17:59 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.60.12    Driver Version: 527.41       CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA T1200 La...  On   | 00000000:01:00.0 Off |                  N/A |
| N/A   54C    P0    18W /  N/A |      0MiB /  4096MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [22]:
sched_schema = pl.Schema(
    {
        'date': pl.Date,
        'total_items': pl.UInt8,
        'total_events': pl.UInt8,
        'total_games': pl.UInt8,
        'total_games_in_progress': pl.UInt8,
        'game_pk': pl.UInt32,
        'game_guid': pl.String,
        'link': pl.String,
        'game_type': pl.String,
        'season': pl.UInt16,
        'game_date': pl.Datetime(),
        'official_date': pl.Date,
        'is_tie': pl.Boolean,
        'game_number': pl.UInt8,
        'public_facing': pl.Boolean,
        'double_header': pl.String,
        'gameday_type': pl.String,
        'tiebreaker': pl.Boolean,
        'calendar_event_id': pl.String,
        'season_display': pl.UInt16,
        'day_night': pl.String,
        'description': pl.String,
        'scheduled_innings': pl.UInt8,
        'reverse_home_away_status': pl.Boolean,
        'inning_break_length': pl.UInt16,
        'games_in_series': pl.UInt8,
        'series_game_number': pl.UInt8,
        'series_description': pl.String,
        'record_source': pl.String,
        'if_necessary': pl.Boolean,
        'if_necessary_description': pl.String,
        'status_abstract_game_state': pl.String,
        'status_coded_game_state': pl.String,
        'status_detailed_state': pl.String,
        'status_status_code': pl.String,
        'status_start_time_tbd': pl.Boolean,
        'status_abstract_game_code': pl.String,
        'teams_away_score': pl.UInt8,
        'teams_away_is_winner': pl.Boolean,
        'teams_away_split_squad': pl.Boolean,
        'teams_away_series_number': pl.UInt8,
        'teams_away_team_id': pl.UInt16,
        'teams_away_team_name': pl.String,
        'teams_away_team_link': pl.String,
        'teams_away_league_record_wins': pl.UInt16,
        'teams_away_league_record_losses': pl.UInt16,
        'teams_away_league_record_ties': pl.UInt16,
        'teams_away_league_record_pct': pl.Float64,
        'teams_home_score': pl.UInt8,
        'teams_home_is_winner': pl.Boolean,
        'teams_home_split_squad': pl.Boolean,
        'teams_home_series_number': pl.UInt8,
        'teams_home_team_id': pl.UInt16,
        'teams_home_team_name': pl.String,
        'teams_home_team_link': pl.String,
        'teams_home_league_record_wins': pl.UInt16,
        'teams_home_league_record_losses': pl.UInt16,
        'teams_home_league_record_ties': pl.UInt16,
        'teams_home_league_record_pct': pl.Float64,
        'venue_id': pl.UInt32,
        'venue_name': pl.String,
        'venue_link': pl.String,
        'content_link': pl.String,
        'status_reason': pl.String,
        'reschedule_date': pl.Datetime(),
        'reschedule_game_date': pl.Date,
        'resume_date': pl.Datetime(),
        'resume_game_date': pl.Date,
        'resumed_from': pl.Datetime(),
        'resumed_from_date': pl.Date,
        'rescheduled_from': pl.Datetime(),
        'rescheduled_from_date': pl.Date,
        'events': pl.String
    }
)

In [23]:
sc_schema = pl.Schema(
    {
        'pitch_type': pl.String,
        'game_date': pl.Date,
        'release_speed': pl.Float64,
        'release_pos_x': pl.Float64,
        'release_pos_z': pl.Float64,
        'player_name': pl.String,
        'batter': pl.UInt32,
        'pitcher': pl.UInt32,
        'events': pl.String,
        'description': pl.String,
        'spin_dir': pl.UInt16,
        'spin_rate_deprecated': pl.UInt16,
        'break_angle_deprecated': pl.UInt16,
        'break_length_deprecated': pl.Float64,
        'zone': pl.String,
        'des': pl.String,
        'game_type': pl.String,
        'stand': pl.String,
        'p_throws': pl.String,
        'home_team': pl.String,
        'away_team': pl.String,
        'type': pl.String,
        'hit_location': pl.String,
        'bb_type': pl.String,
        'balls': pl.UInt8,
        'strikes': pl.UInt8,
        'game_year': pl.UInt16,
        'pfx_x': pl.Float64,
        'pfx_z': pl.Float64,
        'plate_x': pl.Float64,
        'plate_z': pl.Float64,
        'on_3b': pl.UInt32,
        'on_2b': pl.UInt32,
        'on_1b': pl.UInt32,
        'outs_when_up': pl.UInt8,
        'inning': pl.UInt8,
        'inning_topbot': pl.String,
        'hc_x': pl.Float64,
        'hc_y': pl.Float64,
        'tfs_deprecated': pl.String,
        'tfs_zulu_deprecated': pl.String,
        'umpire': pl.UInt32,
        'sv_id': pl.String,
        'vx0': pl.Float64,
        'vy0': pl.Float64,
        'vz0': pl.Float64,
        'ax': pl.Float64,
        'ay': pl.Float64,
        'az': pl.Float64,
        'sz_top': pl.Float64,
        'sz_bot': pl.Float64,
        'hit_distance_sc': pl.UInt16,
        'launch_speed': pl.Float64,
        'launch_angle': pl.Int16,
        'effective_speed': pl.Float64,
        'release_spin_rate': pl.UInt16,
        'release_extension': pl.Float64,
        'game_pk': pl.UInt32,
        'fielder_2': pl.UInt32,
        'fielder_3': pl.UInt32,
        'fielder_4': pl.UInt32,
        'fielder_5': pl.UInt32,
        'fielder_6': pl.UInt32,
        'fielder_7': pl.UInt32,
        'fielder_8': pl.UInt32,
        'fielder_9': pl.UInt32,
        'release_pos_y': pl.Float64,
        'estimated_ba_using_speedangle': pl.Float64,
        'estimated_woba_using_speedangle': pl.Float64,
        'woba_value': pl.Float64,
        'woba_denom': pl.UInt8,
        'babip_value': pl.UInt8,
        'iso_value': pl.UInt8,
        'launch_speed_angle': pl.UInt8,
        'at_bat_number': pl.UInt16,
        'pitch_number': pl.UInt8,
        'pitch_name': pl.String,
        'home_score': pl.UInt8,
        'away_score': pl.UInt8,
        'bat_score': pl.UInt8,
        'fld_score': pl.UInt8,
        'post_away_score': pl.UInt8,
        'post_home_score': pl.UInt8,
        'post_bat_score': pl.UInt8,
        'post_fld_score': pl.UInt8,
        'if_fielding_alignment': pl.String,
        'of_fielding_alignment': pl.String,
        'spin_axis': pl.UInt16,
        'delta_home_win_exp': pl.Float64,
        'delta_run_exp': pl.Float64,
        'bat_speed': pl.Float64,
        'swing_length': pl.Float64,
        'estimated_slg_using_speedangle': pl.Float64,
        'delta_pitcher_run_exp': pl.Float64,
        'hyper_speed': pl.Float64,
        'home_score_diff': pl.Int8,
        'bat_score_diff': pl.Int8,
        'home_win_exp': pl.Float64,
        'bat_win_exp': pl.Float64,
        'age_pit_legacy': pl.UInt8,
        'age_bat_legacy': pl.UInt8,
        'age_pit': pl.UInt8,
        'age_bat': pl.UInt8,
        'n_thruorder_pitcher': pl.UInt8,
        'n_priorpa_thisgame_player_at_bat': pl.UInt8,
        'pitcher_days_since_prev_game': pl.UInt8,
        'batter_days_since_prev_game': pl.UInt8,
        'pitcher_days_until_next_game': pl.UInt8,
        'batter_days_until_next_game': pl.UInt8,
        'api_break_z_with_gravity': pl.Float64,
        'api_break_x_arm': pl.Float64,
        'api_break_x_batter_in': pl.Float64,
        'arm_angle': pl.Float64,
        'attack_angle': pl.Float64,
        'attack_direction': pl.Float64,
        'swing_path_tilt': pl.Float64,
        'intercept_ball_minus_batter_pos_x_inches': pl.Float64,
        'intercept_ball_minus_batter_pos_y_inches': pl.Float64,
    }
)

In [24]:
sc_schema_clean = pl.Schema(
    {
        'pitch_type': pl.String,
        'game_date': pl.Date,
        'release_speed': pl.Float64,
        'release_pos_x': pl.Float64,
        'release_pos_z': pl.Float64,
        'player_name': pl.String,
        'batter': pl.UInt32,
        'pitcher': pl.UInt32,
        'events': pl.String,
        'description': pl.String,
        'zone': pl.String,
        'des': pl.String,
        'game_type': pl.String,
        'stand': pl.String,
        'p_throws': pl.String,
        'home_team': pl.String,
        'away_team': pl.String,
        'type': pl.String,
        'hit_location': pl.String,
        'bb_type': pl.String,
        'balls': pl.UInt8,
        'strikes': pl.UInt8,
        'game_year': pl.UInt16,
        'pfx_x': pl.Float64,
        'pfx_z': pl.Float64,
        'plate_x': pl.Float64,
        'plate_z': pl.Float64,
        'on_3b': pl.UInt32,
        'on_2b': pl.UInt32,
        'on_1b': pl.UInt32,
        'outs_when_up': pl.UInt8,
        'inning': pl.UInt8,
        'inning_topbot': pl.String,
        'hc_x': pl.Float64,
        'hc_y': pl.Float64,
        'vx0': pl.Float64,
        'vy0': pl.Float64,
        'vz0': pl.Float64,
        'ax': pl.Float64,
        'ay': pl.Float64,
        'az': pl.Float64,
        'sz_top': pl.Float64,
        'sz_bot': pl.Float64,
        'hit_distance_sc': pl.UInt16,
        'launch_speed': pl.Float64,
        'launch_angle': pl.Int16,
        'effective_speed': pl.Float64,
        'release_spin_rate': pl.UInt16,
        'release_extension': pl.Float64,
        'game_pk': pl.UInt32,
        'fielder_2': pl.UInt32,
        'fielder_3': pl.UInt32,
        'fielder_4': pl.UInt32,
        'fielder_5': pl.UInt32,
        'fielder_6': pl.UInt32,
        'fielder_7': pl.UInt32,
        'fielder_8': pl.UInt32,
        'fielder_9': pl.UInt32,
        'release_pos_y': pl.Float64,
        'estimated_ba_using_speedangle': pl.Float64,
        'estimated_woba_using_speedangle': pl.Float64,
        'woba_value': pl.Float64,
        'woba_denom': pl.UInt8,
        'babip_value': pl.UInt8,
        'iso_value': pl.UInt8,
        'launch_speed_angle': pl.UInt8,
        'at_bat_number': pl.UInt16,
        'pitch_number': pl.UInt8,
        'pitch_name': pl.String,
        'home_score': pl.UInt8,
        'away_score': pl.UInt8,
        'bat_score': pl.UInt8,
        'fld_score': pl.UInt8,
        'post_away_score': pl.UInt8,
        'post_home_score': pl.UInt8,
        'post_bat_score': pl.UInt8,
        'post_fld_score': pl.UInt8,
        'if_fielding_alignment': pl.String,
        'of_fielding_alignment': pl.String,
        'spin_axis': pl.UInt16,
        'delta_home_win_exp': pl.Float64,
        'delta_run_exp': pl.Float64,
        'bat_speed': pl.Float64,
        'swing_length': pl.Float64,
        'estimated_slg_using_speedangle': pl.Float64,
        'delta_pitcher_run_exp': pl.Float64,
        'hyper_speed': pl.Float64,
        'home_score_diff': pl.Int8,
        'bat_score_diff': pl.Int8,
        'home_win_exp': pl.Float64,
        'bat_win_exp': pl.Float64,
        'age_pit_legacy': pl.UInt8,
        'age_bat_legacy': pl.UInt8,
        'age_pit': pl.UInt8,
        'age_bat': pl.UInt8,
        'n_thruorder_pitcher': pl.UInt8,
        'n_priorpa_thisgame_player_at_bat': pl.UInt8,
        'pitcher_days_since_prev_game': pl.UInt8,
        'batter_days_since_prev_game': pl.UInt8,
        'pitcher_days_until_next_game': pl.UInt8,
        'batter_days_until_next_game': pl.UInt8,
        'api_break_z_with_gravity': pl.Float64,
        'api_break_x_arm': pl.Float64,
        'api_break_x_batter_in': pl.Float64,
        'arm_angle': pl.Float64,
        'attack_angle': pl.Float64,
        'attack_direction': pl.Float64,
        'swing_path_tilt': pl.Float64,
        'intercept_ball_minus_batter_pos_x_inches': pl.Float64,
        'intercept_ball_minus_batter_pos_y_inches': pl.Float64,
    }
)


In [34]:
def clean_schedules(schema):
    for f in schedule.glob('schedule_????.csv'):
        df = pd.read_csv(f)
        map_dict = {'N': False, 'Y': True}

        df['tiebreaker'] = df['tiebreaker'].map(map_dict)

        df['if_necessary'] = df['if_necessary'].map(map_dict)
        for col, dtype in zip(sched_schema.names(), sched_schema.dtypes()):
            if col not in df.columns:
                df[col] = np.nan

            if dtype == pl.UInt8:
                df[col] = df[col].astype('Int64')
            elif dtype == pl.UInt16:
                df[col] = df[col].astype('Int64')
            elif dtype == pl.UInt32:
                df[col] = df[col].astype('Int64')
            elif dtype == pl.UInt64:
                df[col] = df[col].astype('Int64')
        df[schema.names()].to_csv(schedule/('clean_'+f.name), index=False)

        df = pl.scan_csv(schedule/('clean_'+f.name))
        df = df.filter(
            ~pl.col('is_tie').is_null()
        ).with_columns(
            pl.col('description').fill_null('no description'),
            pl.col('inning_break_length').fill_null(strategy='backward'),
            pl.col('games_in_series').fill_null(strategy='zero'),
            pl.col('series_game_number').fill_null(strategy='zero'),
            pl.col('teams_away_is_winner').fill_null(False),
            pl.col('teams_home_is_winner').fill_null(False),
            pl.col('teams_away_series_number').fill_null(strategy='zero'),
            pl.col('teams_home_series_number').fill_null(strategy='zero'),
        )
        df.collect(engine = 'gpu').write_csv(schedule/('clean_'+f.name))
clean_schedules(sched_schema)

In [35]:
def clean_statcast(schema):
    for f in statcast.glob('statcast_????.csv'):
        df = pl.scan_csv(f, schema = schema)
        df = df.with_columns(
            pl.col('events').fill_null(pl.col('description')),
            pl.col('des').fill_null(pl.col('description')),
            pl.when(
                pl.col('hit_location').is_null() &
                (pl.col('events') == 'home_run')
            ).then(
                pl.col('hit_location').fill_null("HR"),
            ).when(
                pl.col('hit_location').is_null() &
                ((pl.col('events') == 'double') & pl.col('des').str.contains('ground-rule double'))
            ).then(
                pl.col('hit_location').fill_null("GRD"),
            ).when(
                pl.col('hit_location').is_null() &
                pl.col('des').str.contains('fan interference') &
                ~pl.col('des').str.contains('ground-rule double')
            ).then(
                pl.col('hit_location').fill_null("FAN"),
            ).when(
                pl.col('hit_location').is_null() &
                (pl.col('type') == 'X')
            ).then(
                pl.col('hit_location').fill_null("UNKNOWN"),
            ).otherwise(
                pl.col('hit_location').fill_null("NO_HIT")
            ),
            pl.col('bb_type').fill_null("not_in_play"),
            cs.matches('on_[1-3]b').fill_null(strategy='zero'),
        ).select(
            pl.exclude(
                'spin_dir',
                'spin_rate_deprecated',
                'break_angle_deprecated',
                'break_length_deprecated',
                'tfs_zulu_deprecated',
                'tfs_deprecated',
                'umpire',
                'sv_id'
            )
        )

        df.collect(engine = 'gpu').write_csv(statcast/('clean_'+f.name))
clean_statcast(sc_schema)

In [27]:
df_sch = pl.scan_csv(schedule/'clean_schedule_????.csv', schema = sched_schema)
df_sc = pl.scan_csv(statcast/'clean_statcast_????.csv', schema = sc_schema_clean)

In [29]:
from polars import GPUEngine
df_sch_gpu = df_sch.with_columns(pl.col(pl.Categorical).cast(pl.String))
df_sch_gpu.collect(engine=GPUEngine(raise_on_fail=True)).null_count()

date,total_items,total_events,total_games,total_games_in_progress,game_pk,game_guid,link,game_type,season,game_date,official_date,is_tie,game_number,public_facing,double_header,gameday_type,tiebreaker,calendar_event_id,season_display,day_night,description,scheduled_innings,reverse_home_away_status,inning_break_length,games_in_series,series_game_number,series_description,record_source,if_necessary,if_necessary_description,status_abstract_game_state,status_coded_game_state,status_detailed_state,status_status_code,status_start_time_tbd,status_abstract_game_code,teams_away_score,teams_away_is_winner,teams_away_split_squad,teams_away_series_number,teams_away_team_id,teams_away_team_name,teams_away_team_link,teams_away_league_record_wins,teams_away_league_record_losses,teams_away_league_record_ties,teams_away_league_record_pct,teams_home_score,teams_home_is_winner,teams_home_split_squad,teams_home_series_number,teams_home_team_id,teams_home_team_name,teams_home_team_link,teams_home_league_record_wins,teams_home_league_record_losses,teams_home_league_record_ties,teams_home_league_record_pct,venue_id,venue_name,venue_link,content_link,status_reason,reschedule_date,reschedule_game_date,resume_date,resume_game_date,resumed_from,resumed_from_date,rescheduled_from,rescheduled_from_date,events
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,30141,30627,30627,30591,30591,30591,30591,30139,30139,30627


In [16]:
df_sc.describe()

statistic,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,vx0,…,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
str,str,str,f64,f64,f64,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,…,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""7662949""","""7799380""",7.665475e6,7.664602e6,7.664602e6,"""7799380""",7.79938e6,7.79938e6,"""7799380""","""7799380""","""7664322""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""",7.79938e6,7.79938e6,7.79938e6,7.664607e6,7.664751e6,7.664322e6,7.664322e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,"""7799380""",1.357832e6,1.357832e6,7.664812e6,…,7.79938e6,7.79938e6,7.79938e6,7.79938e6,"""7581862""","""7581862""",6.923909e6,7.799348e6,7.769841e6,811081.0,811081.0,1.291065e6,7.769841e6,2.290562e6,7.79938e6,7.79938e6,7.79935e6,7.79935e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,7.207525e6,7.46241e6,7.236308e6,7.469149e6,7.664604e6,7.664607e6,7.664607e6,3.786564e6,811081.0,811081.0,811075.0,810175.0,810175.0
"""null_count""","""136431""","""0""",133905.0,134778.0,134778.0,"""0""",0.0,0.0,"""0""","""0""","""135058""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,134773.0,134629.0,135058.0,135058.0,0.0,0.0,0.0,0.0,0.0,"""0""",6.441548e6,6.441548e6,134568.0,…,0.0,0.0,0.0,0.0,"""217518""","""217518""",875471.0,32.0,29539.0,6.988299e6,6.988299e6,6.508315e6,29539.0,5.508818e6,0.0,0.0,30.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,591855.0,336970.0,563072.0,330231.0,134776.0,134773.0,134773.0,4.012816e6,6.988299e6,6.988299e6,6.988305e6,6.989205e6,6.989205e6
"""mean""",null,"""2020-07-31 08:24:19.807000""",88.868149,-0.793726,5.863429,null,581309.795305,582378.317676,null,null,null,null,null,null,null,null,null,null,null,null,0.87748,0.887378,2020.080001,-0.125932,0.652869,0.028987,2.27276,55557.999415,109682.229661,176925.861167,0.982177,4.977218,null,126.678492,122.958997,2.343546,…,2.36213,2.230364,2.292245,2.300249,null,null,178.014429,0.000257,0.000148,69.588397,7.214767,0.544226,-0.000148,91.886702,-0.132085,-0.038833,0.508062,0.513071,28.504961,28.13189,29.026521,28.663105,1.499132,1.515888,5.815862,1.757308,5.927575,1.747634,2.300268,0.376953,-0.113606,38.874974,9.160892,-0.676799,32.332275,37.08873,29.905032
"""std""",null,null,6.058157,1.895897,0.521578,null,85648.870192,82087.405421,null,null,null,null,null,null,null,null,null,null,null,null,0.967923,0.827612,3.242158,0.885809,0.743075,0.860721,0.962367,173183.027611,230889.490157,271649.913661,0.817723,2.632737,null,40.250585,42.304868,5.952344,…,2.64996,2.586684,2.580776,2.657341,null,null,70.552226,0.028289,0.227766,8.973566,1.000515,0.698952,0.227766,6.133604,3.191323,3.193819,0.293245,0.293065,3.736695,3.73555,3.744963,3.757125,0.727544,1.272633,7.572445,4.029154,7.909412,4.024197,1.11242,0.811432,0.887474,12.931691,11.800878,20.48010

In [31]:
df_sc.filter(
    ~pl.col('hit_location').is_null() &
    (pl.col('type') != 'X') &
    (pl.col('hit_location') == "2") &
    (pl.col('events') != 'strikeout') &
    (pl.col('bb_type') != 'not_in_play')
).select(
    'des',
    'events',
    'description',
    'type',
    'hit_location',
    'bb_type'
).collect(engine = 'gpu')

des,events,description,type,hit_location,bb_type
str,str,str,str,str,str
"""Lorenzo Cain reaches on catche…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Tommy La Stella reaches on cat…","""catcher_interf""","""hit_into_play""","""S""","""2""","""line_drive"""
"""Nick Senzel reaches on catcher…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Josh Reddick reaches on catche…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Cavan Biggio reaches on catche…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
…,…,…,…,…,…
"""Travis Jankowski reaches on ca…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Sal Frelick reaches on catcher…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Kerry Carpenter reaches on cat…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""


In [21]:
df_sc.filter(
    pl.col('hit_location').is_null()
).group_by('type').len().collect()

type,len
cat,u32


In [ ]:
df_sc.select(
    cs.matches('hc_[xy]'),
    'hit_distance_sc'
).drop_nulls().collect().sample(1000)


In [ ]:
df_sc.with_columns(
    (np.atan(
        (pl.col('hc_x')-125.42)/(198.27 - pl.col('hc_y'))
    )* 180 /np.pi*.5).alias('spray_angle')
).drop_nulls().select(
    cs.matches('hc_[xy]'),
    'spray_angle'
).describe()

In [ ]:
df_sc.filter(
    pl.col('hc_x').is_null().xor(pl.col('hit_distance_sc').is_null()) &
    (pl.col('des') != 'foul')
).select(
    'des',
    'type',
    'description',
    cs.matches('hc_[xy]'),
    'hit_distance_sc',
    cs.matches('launch'),
    'bat_speed'
).describe()

In [ ]:
df = df_sc.filter(
    pl.col('type') == 'X'
).select(
    cs.matches('hc_[xy]'),
    cs.matches('distance'),
    pl.col('hit_location'),
    cs.matches('launch'),
    'bat_speed'
).drop_nulls().collect().sample(100000).to_dummies(cs.categorical())

In [ ]:

#y = ['hc_x','hc_y']
y = ['hit_distance_sc']
X = [col for col in df.columns if col not in y]


In [ ]:


from sklearn.model_selection import train_test_split as tts
X_train, X_test, y_train, y_test = tts(df[X], df[y], test_size=.2)

In [31]:
from sklearn.linear_model import LinearRegression as lr
from sklearn.preprocessing import PolynomialFeatures as poly
from sklearn.preprocessing import StandardScaler as ss
from sklearn.pipeline import Pipeline

ppl = Pipeline([('Scaler', ss()), ('PolynomialFeatures', poly(3)), ('LinearRegression', lr())])
ppl.fit(X_train, y_train)
ppl.score(X_test, y_test)

0.8250847982540161

In [20]:
from sklearn.neighbors import KNeighborsRegressor as knr
from sklearn.preprocessing import MinMaxScaler as mms
ppl = Pipeline([('Scaler', ss()), ('PolynomialFeatures', poly(2)), ('Regressor', knr())])

ppl.fit(X_train, y_train)
ppl.score(X_test, y_test)


0.7896825855835184

In [89]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict


class RegressorFeature(BaseEstimator, TransformerMixin):
    """
    Wraps a regressor and exposes its predictions as a single feature column.
    Uses cross_val_predict during fit to avoid leakage, then re-fits on full data
    for use at transform time.
    """

    def __init__(self, estimator, cv=5):
        self.estimator = estimator
        self.cv = cv

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        # Re-fit on full data so transform() works on new data
        self.estimator_.fit(X, y)
        return self

    def transform(self, X):
        preds = self.estimator_.predict(X)
        return preds.reshape(-1, 2)

    def fit_transform(self, X, y=None, **fit_params):
        self.estimator_ = clone(self.estimator)
        # Use OOF predictions during training to prevent leakage
        oof_preds = cross_val_predict(self.estimator_, X, y, cv=self.cv)
        self.estimator_.fit(X, y)
        return oof_preds.reshape(-1, 2)

In [93]:
from xgboost import XGBRegressor as xgb
from sklearn.model_selection import GridSearchCV as gcv
from sklearn.pipeline import FeatureUnion as union
ppl = Pipeline([('Scaler', mms()),
                ('PolynomialFeatures', poly(degree=3)),
                ('Scaler2', mms()),
                ('PCA', PCA(n_components=50)),
                ("features", union([
                   # Original features (scaled)
                    ("original", ss()),
                    # New meta-feature: Ridge predictions
                    ("ridge_pred", RegressorFeature(estimator=knr(n_neighbors=5), cv=5)),
                 ])),
                ('Regressor', xgb(n_estimators=200, max_leaves=4, device='cuda'))])

cv = gcv(ppl,
         param_grid={
             'PolynomialFeatures__degree': [1,2],
             'PCA__n_components': [5,10,15],
             'Regressor__n_estimators': [25,50]

         }, cv = 3, verbose=3)

cv.fit(X_train, y_train)
cv.score(X_test, y_test)


Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV 1/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=25;, score=0.812 total time=   0.9s
[CV 2/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=25;, score=0.816 total time=   0.8s
[CV 3/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=25;, score=0.817 total time=   0.7s
[CV 1/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=50;, score=0.815 total time=   0.9s
[CV 2/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=50;, score=0.819 total time=   0.8s
[CV 3/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=50;, score=0.819 total time=   0.7s
[CV 1/3] END PCA__n_components=5, PolynomialFeatures__degree=2, Regressor__n_estimators=25;, score=0.812 total time=   0.7s
[CV 2/3] END PCA__n_components=5, PolynomialFeatures__degree=2, Regress

0.8179455995559692

In [85]:
cv.best_params_

{'PCA__n_components': 50,
 'PolynomialFeatures__degree': 3,
 'Regressor__n_estimators': 200}

In [44]:
from sklearn.neural_network import MLPRegressor as mlp
from sklearn.decomposition import PCA
ppl = Pipeline([('Scaler', mms()),
                ('PolynomialFeatures', poly(3)),
                ('Scaler2', mms()),('PCA', PCA(400)),
                ('Regressor', mlp(hidden_layer_sizes=(18, 18),
                                  learning_rate='adaptive',
                                  max_iter=1000,
                                  verbose = True,
                                  early_stopping=True,
                                  n_iter_no_change=50,))])

ppl.fit(X_train, y_train)
ppl.score(X_test, y_test)

Iteration 1, loss = 5862.62822129
Validation score: -0.216011
Iteration 2, loss = 527.39136294
Validation score: 0.473608
Iteration 3, loss = 361.56313004
Validation score: 0.704010
Iteration 4, loss = 191.99656385
Validation score: 0.810901
Iteration 5, loss = 157.40957688
Validation score: 0.822590
Iteration 6, loss = 151.36522375
Validation score: 0.826908
Iteration 7, loss = 148.76438153
Validation score: 0.828938
Iteration 8, loss = 147.37053712
Validation score: 0.829668
Iteration 9, loss = 146.55653512
Validation score: 0.829881
Iteration 10, loss = 146.07692676
Validation score: 0.830875
Iteration 11, loss = 145.61044356
Validation score: 0.832200
Iteration 12, loss = 145.17588414
Validation score: 0.832603
Iteration 13, loss = 144.96035354
Validation score: 0.832967
Iteration 14, loss = 144.73674371
Validation score: 0.832639
Iteration 15, loss = 144.61044676
Validation score: 0.833356
Iteration 16, loss = 144.40539249
Validation score: 0.833117
Iteration 17, loss = 144.191611

0.8273828712882502

In [95]:
pd.concat([pd.DataFrame(ppl.predict(X_test)), pd.DataFrame(y_test)], axis=1)

,0,1,0,1
0,116.308687,122.801529,51.85,127.94
1,112.537247,149.804064,116.32,149.52
2,114.485353,155.260477,115.05,159.04
3,64.407365,90.625959,70.55,94.31
4,112.516453,149.452030,114.61,145.00
...,...,...,...,...
19995,110.410314,150.067502,106.59,150.63
19996,102.816344,165.397289,109.08,174.53
19997,111.955658,150.672915,112.82,163.35
19998,112.260008,154.040449,138.78,150.90


In [54]:
from sklearn.linear_model import MultiTaskElasticNetCV as ecv
regr = ecv(cv=10, verbose=True)
regr.fit(X_train, y_train)
regr.score(X_test, y_test)

........................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

0.41420841843736295

In [56]:
regr.coef_

array([[-0.0053954 ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        , -0.74594874,  0.        ,  0.789843  ,
         0.        ,  0.        ,  0.        ,  0.        , -0.03319168,
         0.05472859,  0.        ],
       [-0.2798944 ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        , -0.00562719, -0.        ,  0.00800857,
         0.        ,  0.        ,  0.        ,  0.        , -0.52633608,
         0.27489877,  0.        ]])

In [17]:
df_sc.group_by(
    'game_pk'
).agg(
    pl.col('pitch_type').is_null().sum().alias('null_pitch_count'),
    pl.col('game_pk').len().alias('total_pitches'),
    pl.col('game_year').max()
).group_by(
    (pl.col('null_pitch_count')/pl.col('total_pitches') == 1).alias('epic_statcast_failure')
).agg(
    pl.col('null_pitch_count').sum()
).collect()

epic_statcast_failure,null_pitch_count
bool,u32
false,30381
true,106050


In [18]:
import altair as alt
alt.data_transformers.enable("vegafusion")
df = df_sc.group_by(
    'game_pk'
).agg(
    pl.col('release_speed').is_null().sum().alias('null_count'),
    pl.col('game_pk').len().alias('total_pitches'),
    pl.col('game_year').max()
).with_columns(
    (pl.col('null_count')/pl.col('total_pitches')).alias('null_fraction'),
    (pl.col('null_count')/pl.col('total_pitches') == 1).alias('epic_failure')
).filter(
    ~pl.col('epic_failure')
).collect()



In [19]:

chart = alt.Chart(
    df
).mark_bar(

).encode(
    alt.X(
        "null_fraction:Q",
        bin=True,
    ),
    alt.Y('count()').scale(type = 'log')
)
chart

alt.Chart(...)